# Solutions · 9. Process monitoring with Hotelling's T² and the Q residual

Worked solutions to the three exercises of [Lecture 9](../../notebooks/part3_chemometrics/09_multivariate_process_monitoring_hotelling_t2_and_q.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Orthogonal Subspace Decomposition & Exact Hotelling T^2 Mean Identity")
rng = np.random.default_rng(42)
N, d, k = 80, 8, 2
# 2 latent process drivers (throughput u1, jacket temperature u2) governing 8 plant sensors:
U_noc = rng.normal(0.0, 1.0, size=(N, k))
A_mix = rng.normal(0.0, 1.0, size=(k, d))
X_noc = U_noc @ A_mix + rng.normal(0.0, 0.06, size=(N, d))

pca_mspc = dim_reduction.pca_svd(X_noc, n_components=k)
t2_noc = pca_mspc["hotelling_t2"]
q_noc = pca_mspc["q_spe"]

expected_t2_mean = k * (N - 1.0) / N
print(f"Empirical mean(T^2) across N={N} NOC samples : {np.mean(t2_noc):.8f}")
print(f"Exact theoretical identity k*(N-1)/N         : {expected_t2_mean:.8f}")
print(f"NOC 99th-percentile control limits           : T^2_lim = {np.percentile(t2_noc, 99):.3f}, Q_lim = {np.percentile(q_noc, 99):.5f}")
assert np.isclose(np.mean(t2_noc), expected_t2_mean, rtol=1e-13)
heading("Demonstration 2: Idempotence & Mutual Annihilation of Model and Residual Projectors")
P = pca_mspc["loadings"]
mu = pca_mspc["mean"]
lam = pca_mspc["eigenvalues"]
Pi_model = P @ P.T
Pi_res = np.eye(d) - Pi_model

Xc_noc = X_noc - mu
X_hat_noc = Xc_noc @ Pi_model
E_res_noc = Xc_noc @ Pi_res
pythagoras_err = float(np.max(np.abs(np.sum(Xc_noc ** 2, axis=1) - (np.sum(X_hat_noc ** 2, axis=1) + q_noc))))

print(f"trace(Pi_model) = {np.trace(Pi_model):.1f} (dim = k = {k}) | trace(Pi_res) = {np.trace(Pi_res):.1f} (dim = d - k = {d-k})")
print(f"Projector orthogonality ||Pi_model @ Pi_res||_max : {np.max(np.abs(Pi_model @ Pi_res)):.2e}")
print(f"Pythagorean energy decomposition max residual     : {pythagoras_err:.2e}")
assert np.allclose(Pi_model @ Pi_res, 0.0, atol=1e-14) and pythagoras_err < 1e-13
heading("Demonstration 3: Diagnosing Throughput Surges (High T^2) vs. Sensor Drift (High Q_SPE)")
t2_limit = float(np.percentile(t2_noc, 99))
q_limit = float(np.percentile(q_noc, 99))

# Event A: 6-sigma throughput surge along latent variable 1 (obeys physical sensor correlations):
x_surge = mu + (np.array([6.0, 0.0]) @ A_mix)
xc_surge = x_surge - mu
t_surge = xc_surge @ P
t2_surge = float(np.sum((t_surge ** 2) / lam))
q_surge = float(np.sum((xc_surge - t_surge @ P.T) ** 2))

# Event B: Single sensor #4 drifts by +3.5 units (breaks physical correlation across the 8 sensors):
x_drift = mu.copy()
x_drift[3] += 3.5
xc_drift = x_drift - mu
t_drift = xc_drift @ P
t2_drift = float(np.sum((t_drift ** 2) / lam))
q_drift = float(np.sum((xc_drift - t_drift @ P.T) ** 2))

print(f"Normal 99% Limits : T^2_lim = {t2_limit:6.2f}         | Q_lim = {q_limit:.4f}")
print(f"Throughput Surge  : T^2     = {t2_surge:6.2f} (ALARM) | Q     = {q_surge:.4f} (Normal - correlations intact)")
print(f"Sensor #4 Drift   : T^2     = {t2_drift:6.2f}         | Q     = {q_drift:.4f} (Q ALARM: {q_drift/q_limit:.1f}x limit)")
assert t2_surge > 3.0 * t2_limit and q_surge < q_limit
assert q_drift > 20.0 * q_limit
heading("Demonstration 4: Q-Residual Sensor Contribution Plot & Drift Sensitivity Sweep")
e_drift = xc_drift - t_drift @ P.T
q_contributions = e_drift ** 2
culprit_sensor = int(np.argmax(q_contributions))
print("Q-residual sensor contributions e_j^2:", np.round(q_contributions, 3))
print(f"Isolated faulty sensor index: {culprit_sensor} (Sensor #{culprit_sensor + 1} accounts for {100*q_contributions[culprit_sensor]/q_drift:.1f}% of Q)")
assert culprit_sensor == 3

print(f"\n{'Drift dX_4':>10s} | {'Hotelling T^2':>13s} | {'Q_SPE Residual':>14s} | {'Q / Q_limit':>12s} | {'Top Contributor':>15s}")
print("-" * 74)
for d_val in [0.0, 0.25, 0.5, 1.5, 3.5]:
    x_sw = mu.copy()
    x_sw[3] += d_val
    xc_s = x_sw - mu
    ts = xc_s @ P
    es = xc_s - ts @ P.T
    qs = float(np.sum(es ** 2))
    t2s = float(np.sum((ts ** 2) / lam))
    top_s = f"Sensor #{int(np.argmax(es**2))+1}" if d_val > 0 else "None"
    print(f"{d_val:10.2f} | {t2s:13.3f} | {qs:14.4f} | {qs/q_limit:12.1f}x | {top_s:>15s}")
heading("Inside the Algorithm: By-Hand Hotelling T^2 & Q_SPE vs. the Library")
T_hand = Xc_noc @ P
t2_hand = np.sum((T_hand ** 2) / lam[None, :], axis=1)
q_hand = np.sum((Xc_noc - T_hand @ P.T) ** 2, axis=1)
diff_t2 = float(np.max(np.abs(t2_hand - t2_noc)))
diff_q = float(np.max(np.abs(q_hand - q_noc)))
print("by hand (pure NumPy) T^2[:4]:", np.round(t2_hand[:4], 6))
print("library (engml_2_applied)      T^2[:4]:", np.round(t2_noc[:4], 6))
print(f"Max discrepancy T^2 = {diff_t2:.2e}, Q_SPE = {diff_q:.2e}")
assert diff_t2 < 1e-14 and diff_q < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Inject a correlation-breaking fault into Sensor #7 (index `6`, $+4.0$ bias) of the 8-sensor reactor model (`pca_mspc`), compute the residual vector $\mathbf{e} = (\mathbf{I}_8 - \mathbf{P}_2\mathbf{P}_2^\top)(\mathbf{x} - \boldsymbol{\mu})$, and verify that (1) $\mathbf{P}_2^\top \mathbf{e} = \mathbf{0}$ to machine precision ($<10^{-14}$), (2) $Q > 30 Q_{\mathrm{limit}}$, and (3) `np.argmax(e**2) == 6`.

In [2]:
heading("Exercise 1: Sensor #7 Fault Isolation & Subspace Orthogonality Check")
x_f7 = mu.copy()
x_f7[6] += 4.0
xc_f7 = x_f7 - mu
e_f7 = xc_f7 - (xc_f7 @ P) @ P.T
q_f7 = float(np.sum(e_f7 ** 2))

assert np.max(np.abs(P.T @ e_f7)) < 1e-14
assert q_f7 > 30.0 * q_limit and int(np.argmax(e_f7 ** 2)) == 6
print(f"Sensor #7 fault Q = {q_f7:.3f} | Orthogonality ||P^T e||_max = {np.max(np.abs(P.T @ e_f7)):.2e} | Isolated index = {int(np.argmax(e_f7**2))}")


Exercise 1: Sensor #7 Fault Isolation & Subspace Orthogonality Check
Sensor #7 fault Q = 9.345 | Orthogonality ||P^T e||_max = 2.44e-15 | Isolated index = 6


**Discussion.** The residual vector $\mathbf{e} = (\mathbf{I}_8 - \mathbf{P}_2\mathbf{P}_2^\top)\mathbf{x}_c$ is strictly orthogonal to both principal loading vectors, and its squared components $e_j^2$ immediately pinpoint Sensor #7 (index `6`).

---

## Exercise 2

**Problem.** Demonstrate what happens if an engineer chooses `n_components = d = 8` (retaining *all* principal components in the MSPC model): show that $\mathbf{I}_8 - \mathbf{P}_8\mathbf{P}_8^\top = \mathbf{0}$, causing $Q_{\mathrm{SPE}} \equiv 0$ identically even for a sensor failure of $+100$ units.

In [3]:
heading("Exercise 2: Full-Rank PCA Over-Parameterization Failure Audit (Q_SPE == 0)")
pca_full8 = dim_reduction.pca_svd(X_noc, n_components=d)
P8 = pca_full8["loadings"]
e_blind = xc_f7 - (xc_f7 @ P8) @ P8.T
q_blind = float(np.sum(e_blind ** 2))
print(f"When k = d = 8, ||I - P_8 P_8^T||_max = {np.max(np.abs(np.eye(d) - P8 @ P8.T)):.2e} | Faulty Q_SPE = {q_blind:.2e} (Completely blind)")
assert q_blind < 1e-24


Exercise 2: Full-Rank PCA Over-Parameterization Failure Audit (Q_SPE == 0)
When k = d = 8, ||I - P_8 P_8^T||_max = 6.66e-16 | Faulty Q_SPE = 1.48e-29 (Completely blind)


**Discussion.** When $k = d$, the residual subspace $\mathcal{S}_{\mathrm{res}}^\perp$ has dimension $d - d = 0$, so $Q_{\mathrm{SPE}}$ vanishes identically for every vector in $\mathbb{R}^d$. MSPC *requires* $k < d$ so that a non-trivial residual subspace monitors conservation laws.

---

## Exercise 3

**Problem.** Implement a **fault sensor reconstruction** function `reconstruct_faulty_sensor(x_faulty: np.ndarray, fault_idx: int, mean: np.ndarray, loadings: np.ndarray) -> np.ndarray` that replaces the corrupted $j$-th sensor reading with the value $\hat{x}_j$ that minimizes $Q_{\mathrm{SPE}} = \|(\mathbf{I}_d - \mathbf{P}_k\mathbf{P}_k^\top)(\mathbf{x} - \boldsymbol{\mu})\|_2^2$, given in closed form by $\hat{x}_{\mathrm{corr},j} = x_j - e_j / (\mathbf{I}_d - \mathbf{P}_k\mathbf{P}_k^\top)_{jj}$. Verify on `x_drift` (`fault_idx=3`) that it recovers the true normal reading $\mu_3$ and drops $Q_{\mathrm{SPE}}$ back to $0$.

In [4]:
heading("Exercise 3: Implement Subspace-Projected Faulty Sensor Reconstruction")
def reconstruct_faulty_sensor(x_faulty: np.ndarray, fault_idx: int, mean: np.ndarray, loadings: np.ndarray) -> dict:
    """Reconstruct a broken sensor reading by projecting along coordinate j onto the PCA model subspace."""
    x_corr = np.asarray(x_faulty, dtype=float).copy()
    Pi_r = np.eye(len(mean)) - loadings @ loadings.T
    e_vec = Pi_r @ (x_corr - mean)
    x_corr[fault_idx] -= e_vec[fault_idx] / max(Pi_r[fault_idx, fault_idx], 1e-15)
    e_after = Pi_r @ (x_corr - mean)
    return {"x_reconstructed": x_corr, "q_before": float(np.sum(e_vec ** 2)), "q_after": float(np.sum(e_after ** 2))}

rec_s4 = reconstruct_faulty_sensor(x_drift, fault_idx=3, mean=mu, loadings=P)
print(f"Sensor #4 true normal value = {mu[3]:.4f} | Drifted value = {x_drift[3]:.4f} | Subspace reconstructed = {rec_s4['x_reconstructed'][3]:.4f}")
print(f"Q_SPE before reconstruction = {rec_s4['q_before']:.4f} -> Q_SPE after = {rec_s4['q_after']:.2e}")
assert np.isclose(rec_s4["x_reconstructed"][3], mu[3], atol=1e-12) and rec_s4["q_after"] < 1e-24


Exercise 3: Implement Subspace-Projected Faulty Sensor Reconstruction
Sensor #4 true normal value = 0.0132 | Drifted value = 3.5132 | Subspace reconstructed = 0.0132
Q_SPE before reconstruction = 11.0822 -> Q_SPE after = 8.82e-34


**Discussion.** As long as the diagonal residual leverage $(\mathbf{I}_d - \mathbf{P}_k\mathbf{P}_k^\top)_{jj} > 0$, a single broken sensor can be both isolated via $e_j^2$ and analytically reconstructed from the remaining $d-1$ correlated sensors.